In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split, KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from scipy.stats import spearmanr, kendalltau
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor

PROCESSED_DIR = Path("data/processed")
TABLES_DIR = Path("outputs/tables")
RANDOM_STATE = 42
TABLES_DIR.mkdir(parents=True, exist_ok=True)

cards = pd.read_csv(PROCESSED_DIR / "pokemon_priced_features.csv", low_memory=False)
with open(PROCESSED_DIR / "feature_groups.json", encoding="utf-8") as file:
    groups = json.load(file)

intrinsic = groups["intrinsic_features"]
full_features = intrinsic + groups["extrinsic_features"] + groups["extrinsic_interaction_features"]

feature_sets = {
    "Intrinsic only": intrinsic,
    "Intrinsic + extrinsic": full_features
}

pd.DataFrame([
    {"feature_set": name, "feature_count": len(cols), "features": ", ".join(cols)}
    for name, cols in feature_sets.items()
]).to_csv(TABLES_DIR / "model_feature_sets.csv", index=False, encoding="utf-8-sig")

In [2]:
def pipeline_for(X, estimator):
    categorical = X.select_dtypes(exclude=np.number).columns.tolist()
    numeric = X.select_dtypes(include=np.number).columns.tolist()

    preprocessor = ColumnTransformer([
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=True))
        ]), categorical),
        ("num", SimpleImputer(strategy="median"), numeric)
    ])

    return Pipeline([("preprocessor", preprocessor), ("model", estimator)])

def lgbm():
    return LGBMRegressor(
        n_estimators=800, learning_rate=.03, num_leaves=31,
        subsample=.85, colsample_bytree=.85,
        reg_lambda=2, reg_alpha=.1,
        random_state=RANDOM_STATE, n_jobs=-1, verbose=-1
    )

def scores(model_name, feature_set, y_train, train_pred, y_test, test_pred):
    actual = np.expm1(y_test)
    predicted = np.maximum(np.expm1(test_pred), 0)
    train_rmse = mean_squared_error(y_train, train_pred) ** .5
    test_rmse = mean_squared_error(y_test, test_pred) ** .5

    return {
        "model": model_name,
        "feature_set": feature_set,
        "train_log_MAE": mean_absolute_error(y_train, train_pred),
        "test_log_MAE": mean_absolute_error(y_test, test_pred),
        "train_log_RMSE": train_rmse,
        "test_log_RMSE": test_rmse,
        "RMSE_gap": test_rmse - train_rmse,
        "train_log_R2": r2_score(y_train, train_pred),
        "test_log_R2": r2_score(y_test, test_pred),
        "test_price_MAE": mean_absolute_error(actual, predicted),
        "test_price_RMSE": mean_squared_error(actual, predicted) ** .5,
        "spearman_rank": spearmanr(y_test, test_pred).correlation,
        "kendall_rank": kendalltau(y_test, test_pred).correlation
    }

def cross_validate(estimator, columns, model_name, feature_set):
    X = cards[columns]
    rows = []
    for fold, (fold_train, fold_val) in enumerate(folds.split(X), 1):
        model = pipeline_for(X.iloc[fold_train], clone(estimator))
        model.fit(X.iloc[fold_train], y.iloc[fold_train])
        row = scores(
            model_name, feature_set,
            y.iloc[fold_train], model.predict(X.iloc[fold_train]),
            y.iloc[fold_val], model.predict(X.iloc[fold_val])
        )
        row["fold"] = fold
        rows.append(row)
    return pd.DataFrame(rows).rename(columns=lambda name: name.replace("test_", "val_"))

def summarise_cv(results, group):
    return results.groupby(group).agg(
        log_RMSE_mean=("val_log_RMSE", "mean"),
        log_RMSE_std=("val_log_RMSE", "std"),
        log_MAE_mean=("val_log_MAE", "mean"),
        log_MAE_std=("val_log_MAE", "std"),
        log_R2_mean=("val_log_R2", "mean"),
        log_R2_std=("val_log_R2", "std"),
        RMSE_gap_mean=("RMSE_gap", "mean"),
        price_MAE_mean=("val_price_MAE", "mean"),
        price_RMSE_mean=("val_price_RMSE", "mean"),
        spearman_mean=("spearman", "mean"),
        kendall_mean=("kendall", "mean")
    ).reset_index()

In [3]:
indices = np.arange(len(cards))
train_idx, test_idx = train_test_split(indices, test_size=.2, random_state=RANDOM_STATE)
y = cards["Log_Value"].reset_index(drop=True)

holdout_rows = []

for feature_set, columns in feature_sets.items():
    X = cards[columns]
    model = pipeline_for(X.iloc[train_idx], lgbm())
    model.fit(X.iloc[train_idx], y.iloc[train_idx])

    holdout_rows.append(scores(
        "LightGBM", feature_set,
        y.iloc[train_idx], model.predict(X.iloc[train_idx]),
        y.iloc[test_idx], model.predict(X.iloc[test_idx])
    ))

holdout_results = pd.DataFrame(holdout_rows)
display(holdout_results)

intrinsic_result = holdout_results.set_index("feature_set").loc["Intrinsic only"]
full_result = holdout_results.set_index("feature_set").loc["Intrinsic + extrinsic"]

holdout_impact = pd.DataFrame([{
    "RMSE_improvement": intrinsic_result["test_log_RMSE"] - full_result["test_log_RMSE"],
    "RMSE_improvement_pct": (intrinsic_result["test_log_RMSE"] - full_result["test_log_RMSE"]) / intrinsic_result["test_log_RMSE"] * 100,
    "R2_improvement": full_result["test_log_R2"] - intrinsic_result["test_log_R2"],
    "price_MAE_improvement": intrinsic_result["test_price_MAE"] - full_result["test_price_MAE"]
}])

display(holdout_impact)
holdout_results.to_csv(TABLES_DIR / "intrinsic_extrinsic_holdout.csv", index=False, encoding="utf-8-sig")
holdout_impact.to_csv(TABLES_DIR / "intrinsic_extrinsic_holdout_impact.csv", index=False, encoding="utf-8-sig")

C:\Users\rayan\PycharmProjects\Dissertation\.venv\Lib\site-packages\sklearn\impute\_base.py:647: UserWarning: Skipping features without any observed values: ['card_number_printed_total' 'card_position_ratio_intrinsic']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
C:\Users\rayan\PycharmProjects\Dissertation\.venv\Lib\site-packages\sklearn\impute\_base.py:647: UserWarning: Skipping features without any observed values: ['card_number_printed_total' 'card_position_ratio_intrinsic']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
C:\Users\rayan\PycharmProjects\Dissertation\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\rayan\PycharmProjects\Dissertation\.venv\Lib\site-packages\sklearn\impute\_base.py:647: UserWarning: Skipping features without any observed 

,model,feature_set,train_log_MAE,test_log_MAE,train_log_RMSE,test_log_RMSE,RMSE_gap,train_log_R2,test_log_R2,test_price_MAE,test_price_RMSE,spearman_rank,kendall_rank
0,LightGBM,Intrinsic only,0.358668,0.378888,0.568473,0.599981,0.031509,0.747720,0.720465,6.527930,48.740998,0.788678,0.615303
1,LightGBM,Intrinsic + extrinsic,0.221738,0.246741,0.365954,0.412865,0.046911,0.895452,0.867634,4.821715,38.608408,0.898872,0.747713


,RMSE_improvement,RMSE_improvement_pct,R2_improvement,price_MAE_improvement
0,0.187116,31.18704,0.147169,1.706215


In [4]:
folds = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
feature_cv = pd.concat([
    cross_validate(lgbm(), columns, "LightGBM", name)
    for name, columns in feature_sets.items()
], ignore_index=True)

cv_metrics = [
    "fold", "train_log_RMSE", "val_log_RMSE", "RMSE_gap", "val_log_MAE", "val_log_R2",
    "val_price_MAE", "val_price_RMSE", "spearman", "kendall"
]
rank_names = {"spearman_rank": "spearman", "kendall_rank": "kendall"}
cv_results = feature_cv.rename(columns=rank_names)[["feature_set"] + cv_metrics]
cv_summary = summarise_cv(cv_results, "feature_set")

display(cv_results)
display(cv_summary)
cv_results.to_csv(TABLES_DIR / "intrinsic_extrinsic_cv_folds.csv", index=False, encoding="utf-8-sig")
cv_summary.to_csv(TABLES_DIR / "intrinsic_extrinsic_cv_summary.csv", index=False, encoding="utf-8-sig")

C:\Users\rayan\PycharmProjects\Dissertation\.venv\Lib\site-packages\sklearn\impute\_base.py:647: UserWarning: Skipping features without any observed values: ['card_number_printed_total' 'card_position_ratio_intrinsic']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
C:\Users\rayan\PycharmProjects\Dissertation\.venv\Lib\site-packages\sklearn\impute\_base.py:647: UserWarning: Skipping features without any observed values: ['card_number_printed_total' 'card_position_ratio_intrinsic']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
C:\Users\rayan\PycharmProjects\Dissertation\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\rayan\PycharmProjects\Dissertation\.venv\Lib\site-packages\sklearn\impute\_base.py:647: UserWarning: Skipping features without any observed 

,feature_set,fold,train_log_RMSE,val_log_RMSE,RMSE_gap,val_log_MAE,val_log_R2,val_price_MAE,val_price_RMSE,spearman,kendall
0,Intrinsic only,1,0.568473,0.599981,0.031509,0.378888,0.720465,6.527930,48.740998,0.788678,0.615303
1,Intrinsic only,2,0.561019,0.625847,0.064828,0.386249,0.708758,7.811705,99.702297,0.806061,0.630887
2,Intrinsic only,3,0.567690,0.600301,0.032611,0.379205,0.713948,6.373505,61.335288,0.802626,0.628406
3,Intrinsic only,4,0.566025,0.612505,0.046480,0.382313,0.705861,6.672284,54.306725,0.799853,0.625944
4,Intrinsic only,5,0.567067,0.604594,0.037527,0.377402,0.706021,6.491301,56.487488,0.804685,0.630503
5,Intrinsic + extrinsic,1,0.365954,0.412865,0.046911,0.246741,0.867634,4.821715,38.608408,0.898872,0.747713
6,Intrinsic + extrinsic,2,0.365270,0.420959,0.055689,0.250082,0.868235,6.136634,91.033358,0.904254,0.753915
7,Intrinsic + extrinsic,3,0.366260,0.415960,0.049700,0.248764,0.862656,4.971414,58.505693,0.905030,0.755341
8,Intrinsic + extrinsic,4,0.364860,0.425359,0.060499,0.251959,0.858145,4.961408,42.749559,0.896135,0.743650
9,Intrinsic + extrinsic,5,0.363027,0.432264,0.069237,0.250241,0.849725,4.908463,47.770740,0.901180,0.749777


,feature_set,log_RMSE_mean,log_RMSE_std,log_MAE_mean,log_MAE_std,log_R2_mean,log_R2_std,RMSE_gap_mean,price_MAE_mean,price_RMSE_mean,spearman_mean,kendall_mean
0,Intrinsic + extrinsic,0.421482,0.007686,0.249557,0.001941,0.861279,0.007648,0.056407,5.159927,55.733551,0.901094,0.750079
1,Intrinsic only,0.608646,0.010861,0.380811,0.003526,0.711011,0.006215,0.042591,6.775345,64.114559,0.800381,0.626209


In [5]:
summary = cv_summary.set_index("feature_set")
intrinsic_cv = summary.loc["Intrinsic only"]
full_cv = summary.loc["Intrinsic + extrinsic"]
rmse_reduction = intrinsic_cv["log_RMSE_mean"] - full_cv["log_RMSE_mean"]
rmse_reduction_pct = rmse_reduction / intrinsic_cv["log_RMSE_mean"] * 100
r2_increase = full_cv["log_R2_mean"] - intrinsic_cv["log_R2_mean"]

extrinsic_impact = pd.DataFrame([{
    "intrinsic_CV_RMSE": intrinsic_cv["log_RMSE_mean"],
    "intrinsic_plus_extrinsic_CV_RMSE": full_cv["log_RMSE_mean"],
    "RMSE_reduction": rmse_reduction,
    "RMSE_reduction_pct": rmse_reduction_pct,
    "intrinsic_CV_R2": intrinsic_cv["log_R2_mean"],
    "intrinsic_plus_extrinsic_CV_R2": full_cv["log_R2_mean"],
    "R2_increase": r2_increase
}])
cv_impact = pd.DataFrame([{
    "CV_RMSE_improvement": rmse_reduction,
    "CV_RMSE_improvement_pct": rmse_reduction_pct,
    "CV_R2_improvement": r2_increase,
    "CV_price_MAE_improvement": intrinsic_cv["price_MAE_mean"] - full_cv["price_MAE_mean"]
}])

display(extrinsic_impact)
display(cv_impact)
extrinsic_impact.to_csv(TABLES_DIR / "extrinsic_impact.csv", index=False, encoding="utf-8-sig")
cv_impact.to_csv(TABLES_DIR / "intrinsic_extrinsic_cv_impact.csv", index=False, encoding="utf-8-sig")

,intrinsic_CV_RMSE,intrinsic_plus_extrinsic_CV_RMSE,RMSE_reduction,RMSE_reduction_pct,intrinsic_CV_R2,intrinsic_plus_extrinsic_CV_R2,R2_increase
0,0.608646,0.421482,0.187164,30.750909,0.711011,0.861279,0.150268


,CV_RMSE_improvement,CV_RMSE_improvement_pct,CV_R2_improvement,CV_price_MAE_improvement
0,0.187164,30.750909,0.150268,1.615418


In [6]:
X = cards[full_features]
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

models = {
    "Dummy mean baseline": DummyRegressor(strategy="mean"),
    "Ridge Regression": Ridge(alpha=10),
    "Random Forest": RandomForestRegressor(
        n_estimators=150, max_depth=28, min_samples_split=4,
        max_features=.8, bootstrap=True, oob_score=True,
        random_state=RANDOM_STATE, n_jobs=-1
    ),
    "Extra Trees": ExtraTreesRegressor(
        n_estimators=250, max_depth=28, min_samples_split=4,
        max_features=.8, random_state=RANDOM_STATE, n_jobs=-1
    ),
    "LightGBM": lgbm(),
    "XGBoost": XGBRegressor(
        n_estimators=600, learning_rate=.03, max_depth=6,
        subsample=.85, colsample_bytree=.85,
        reg_lambda=2, reg_alpha=.1,
        objective="reg:squarederror", random_state=RANDOM_STATE, n_jobs=-1
    )
}

model_rows = []

for name, estimator in models.items():
    if name == "LightGBM":
        model_rows.append(holdout_results.iloc[1].to_dict())
        continue
    model = pipeline_for(X_train, estimator)
    model.fit(X_train, y_train)
    model_rows.append(scores(
        name, "Intrinsic + extrinsic",
        y_train, model.predict(X_train),
        y_test, model.predict(X_test)
    ))

cat_train, cat_test = X_train.copy(), X_test.copy()
cat_cols = cat_train.select_dtypes(exclude=np.number).columns.tolist()
num_cols = cat_train.select_dtypes(include=np.number).columns.tolist()

for col in cat_cols:
    cat_train[col] = cat_train[col].fillna("Unknown").astype(str)
    cat_test[col] = cat_test[col].fillna("Unknown").astype(str)

for col in num_cols:
    median = cat_train[col].median()
    cat_train[col] = cat_train[col].fillna(median)
    cat_test[col] = cat_test[col].fillna(median)

catboost = CatBoostRegressor(
    iterations=1000, learning_rate=.03, depth=8,
    l2_leaf_reg=5, loss_function="RMSE",
    random_seed=RANDOM_STATE, verbose=False
)
catboost.fit(cat_train, y_train, cat_features=cat_cols)

model_rows.append(scores(
    "CatBoost", "Intrinsic + extrinsic",
    y_train, catboost.predict(cat_train),
    y_test, catboost.predict(cat_test)
))

model_results = pd.DataFrame(model_rows).sort_values("test_log_RMSE").reset_index(drop=True)
display(model_results)
model_results.to_csv(TABLES_DIR / "model_comparison.csv", index=False, encoding="utf-8-sig")

C:\Users\rayan\PycharmProjects\Dissertation\.venv\Lib\site-packages\sklearn\impute\_base.py:647: UserWarning: Skipping features without any observed values: ['card_number_printed_total' 'card_position_ratio_intrinsic']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
C:\Users\rayan\PycharmProjects\Dissertation\.venv\Lib\site-packages\sklearn\impute\_base.py:647: UserWarning: Skipping features without any observed values: ['card_number_printed_total' 'card_position_ratio_intrinsic']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
C:\Users\rayan\PycharmProjects\Dissertation\.venv\Lib\site-packages\sklearn\impute\_base.py:647: UserWarning: Skipping features without any observed values: ['card_number_printed_total' 'card_position_ratio_intrinsic']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
C:\Users\rayan\AppData\Local\Temp\ipykernel_23988

,model,feature_set,train_log_MAE,test_log_MAE,train_log_RMSE,test_log_RMSE,RMSE_gap,train_log_R2,test_log_R2,test_price_MAE,test_price_RMSE,spearman_rank,kendall_rank
0,LightGBM,Intrinsic + extrinsic,0.221738,0.246741,0.365954,0.412865,0.046911,0.895452,8.676341e-01,4.821715,38.608408,0.898872,0.747713
1,XGBoost,Intrinsic + extrinsic,0.230708,0.253889,0.373785,0.418300,0.044515,0.890929,8.641261e-01,4.816489,37.091095,0.890976,0.737577
2,Random Forest,Intrinsic + extrinsic,0.122780,0.241781,0.215740,0.418694,0.202954,0.963665,8.638702e-01,4.966690,38.335350,0.912550,0.767647
3,Extra Trees,Intrinsic + extrinsic,0.094396,0.241727,0.176979,0.428060,0.251082,0.975548,8.577114e-01,4.858692,34.911619,0.912129,0.766761
4,CatBoost,Intrinsic + extrinsic,0.243849,0.261130,0.397308,0.428911,0.031603,0.876769,8.571455e-01,5.328161,43.026586,0.892699,0.738367
5,Ridge Regression,Intrinsic + extrinsic,0.402177,0.403275,0.605455,0.605083,-0.000372,0.713827,7.156913e-01,7.339817,50.884483,0.827562,0.657672
6,Dummy mean baseline,Intrinsic + extrinsic,0.847217,0.847051,1.131795,1.134802,0.003007,0.000000,-3.802434e-07,9.842178,60.216128,NaN,NaN


In [7]:
algorithm_results = [feature_cv[feature_cv["feature_set"] == "Intrinsic + extrinsic"]]
for name in ["Extra Trees", "Random Forest"]:
    estimator = clone(models[name])
    if name == "Random Forest":
        estimator.set_params(oob_score=False)
    algorithm_results.append(cross_validate(estimator, full_features, name, "Intrinsic + extrinsic"))

algorithm_cv = pd.concat(algorithm_results, ignore_index=True)
model_cv = algorithm_cv.rename(columns=rank_names)[["model"] + cv_metrics]
model_cv_summary = summarise_cv(model_cv, "model").sort_values("log_RMSE_mean").reset_index(drop=True)
fold_winners = model_cv.loc[
    model_cv.groupby("fold")["val_log_RMSE"].idxmin(),
    ["fold", "model", "val_log_RMSE", "val_log_R2"]
].sort_values("fold").reset_index(drop=True)

display(model_cv)
display(model_cv_summary)
display(fold_winners)
print("RMSE wins:")
print(fold_winners["model"].value_counts())

model_cv.to_csv(TABLES_DIR / "final_model_cv_folds.csv", index=False, encoding="utf-8-sig")
model_cv_summary.to_csv(TABLES_DIR / "final_model_cv_summary.csv", index=False, encoding="utf-8-sig")
fold_winners.to_csv(TABLES_DIR / "final_model_cv_winners.csv", index=False, encoding="utf-8-sig")

C:\Users\rayan\PycharmProjects\Dissertation\.venv\Lib\site-packages\sklearn\impute\_base.py:647: UserWarning: Skipping features without any observed values: ['card_number_printed_total' 'card_position_ratio_intrinsic']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
C:\Users\rayan\PycharmProjects\Dissertation\.venv\Lib\site-packages\sklearn\impute\_base.py:647: UserWarning: Skipping features without any observed values: ['card_number_printed_total' 'card_position_ratio_intrinsic']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
C:\Users\rayan\PycharmProjects\Dissertation\.venv\Lib\site-packages\sklearn\impute\_base.py:647: UserWarning: Skipping features without any observed values: ['card_number_printed_total' 'card_position_ratio_intrinsic']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
C:\Users\rayan\PycharmProjects\Dissertation\.venv

,model,fold,train_log_RMSE,val_log_RMSE,RMSE_gap,val_log_MAE,val_log_R2,val_price_MAE,val_price_RMSE,spearman,kendall
0,LightGBM,1,0.365954,0.412865,0.046911,0.246741,0.867634,4.821715,38.608408,0.898872,0.747713
1,LightGBM,2,0.365270,0.420959,0.055689,0.250082,0.868235,6.136634,91.033358,0.904254,0.753915
2,LightGBM,3,0.366260,0.415960,0.049700,0.248764,0.862656,4.971414,58.505693,0.905030,0.755341
3,LightGBM,4,0.364860,0.425359,0.060499,0.251959,0.858145,4.961408,42.749559,0.896135,0.743650
4,LightGBM,5,0.363027,0.432264,0.069237,0.250241,0.849725,4.908463,47.770740,0.901180,0.749777
5,Extra Trees,1,0.176017,0.429248,0.253231,0.242460,0.856921,4.900222,35.290688,0.911561,0.766053
6,Extra Trees,2,0.176289,0.424495,0.248206,0.244186,0.866013,6.042555,89.444565,0.912533,0.767659
7,Extra Trees,3,0.178167,0.421367,0.243200,0.239918,0.859062,5.179629,59.883862,0.914168,0.770119
8,Extra Trees,4,0.173751,0.446325,0.272574,0.250037,0.843817,5.788420,55.144242,0.903191,0.755284
9,Extra Trees,5,0.170312,0.444963,0.274651,0.246432,0.840766,4.940918,50.379268,0.906959,0.759936


,model,log_RMSE_mean,log_RMSE_std,log_MAE_mean,log_MAE_std,log_R2_mean,log_R2_std,RMSE_gap_mean,price_MAE_mean,price_RMSE_mean,spearman_mean,kendall_mean
0,LightGBM,0.421482,0.007686,0.249557,0.001941,0.861279,0.007648,0.056407,5.159927,55.733551,0.901094,0.750079
1,Random Forest,0.428279,0.013129,0.244840,0.003151,0.856625,0.011792,0.213433,5.481864,60.955645,0.911250,0.765947
2,Extra Trees,0.433280,0.011640,0.244606,0.003859,0.853316,0.010665,0.258372,5.370349,58.028525,0.909682,0.763810


,fold,model,val_log_RMSE,val_log_R2
0,1,LightGBM,0.412865,0.867634
1,2,Random Forest,0.419169,0.869354
2,3,LightGBM,0.415960,0.862656
3,4,LightGBM,0.425359,0.858145
4,5,LightGBM,0.432264,0.849725


RMSE wins:
model
LightGBM         4
Random Forest    1
Name: count, dtype: int64


In [8]:
pair_cv = algorithm_cv[algorithm_cv["model"].isin(["LightGBM", "Random Forest"])].copy()
pair_summary = pair_cv.groupby("model").agg(
    val_log_RMSE_mean=("val_log_RMSE", "mean"),
    val_log_RMSE_std=("val_log_RMSE", "std"),
    val_log_R2_mean=("val_log_R2", "mean"),
    RMSE_gap_mean=("RMSE_gap", "mean"),
    val_price_MAE_mean=("val_price_MAE", "mean"),
    val_price_RMSE_mean=("val_price_RMSE", "mean"),
    spearman_mean=("spearman_rank", "mean"),
    kendall_mean=("kendall_rank", "mean")
).reset_index().sort_values("val_log_RMSE_mean")

display(pair_summary)
pair_cv.to_csv(TABLES_DIR / "lightgbm_vs_random_forest_cv_folds.csv", index=False, encoding="utf-8-sig")
pair_summary.to_csv(TABLES_DIR / "lightgbm_vs_random_forest_cv_summary.csv", index=False, encoding="utf-8-sig")

with open(PROCESSED_DIR / "selected_feature_set.json", "w", encoding="utf-8") as file:
    json.dump({"name": "Intrinsic + extrinsic", "features": full_features}, file, indent=2)

,model,val_log_RMSE_mean,val_log_RMSE_std,val_log_R2_mean,RMSE_gap_mean,val_price_MAE_mean,val_price_RMSE_mean,spearman_mean,kendall_mean
0,LightGBM,0.421482,0.007686,0.861279,0.056407,5.159927,55.733551,0.901094,0.750079
1,Random Forest,0.428279,0.013129,0.856625,0.213433,5.481864,60.955645,0.911250,0.765947
